In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.datasets
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn import metrics

In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

importing the boston house price dataset

In [21]:
house_price_dataset = sklearn.datasets.load_boston()

ImportError: 
`load_boston` has been removed from scikit-learn since version 1.2.

The Boston housing prices dataset has an ethical problem: as
investigated in [1], the authors of this dataset engineered a
non-invertible variable "B" assuming that racial self-segregation had a
positive impact on house prices [2]. Furthermore the goal of the
research that led to the creation of this dataset was to study the
impact of air quality but it did not give adequate demonstration of the
validity of this assumption.

The scikit-learn maintainers therefore strongly discourage the use of
this dataset unless the purpose of the code is to study and educate
about ethical issues in data science and machine learning.

In this special case, you can fetch the dataset from the original
source::

    import pandas as pd
    import numpy as np

    data_url = "http://lib.stat.cmu.edu/datasets/boston"
    raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
    data = np.hstack([raw_df.values[:

In [12]:
from sklearn.datasets import fetch_openml
import pandas as pd
boston = fetch_openml(name="boston", version=1, as_frame=True)
df = boston.frame
df.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


In [13]:
df['price']=boston.target
df.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV,price
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2,36.2


dividing the dataset into dependent and independent features

In [36]:
X=df.iloc[:,:-1]
X=X.astype('float')
X.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1.0,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2.0,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2.0,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3.0,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3.0,222.0,18.7,396.90,5.33,36.2


In [35]:
y=df['price']
y=y.astype('float')
y.head()

,price
0,24.0
1,21.6
2,34.7
3,33.4
4,36.2


splitting the data into training and test data


In [16]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=2)

In [27]:
X_train=X_train.astype('float')
X_test=X_test.astype('float')

In [17]:
print(X.shape,X_train.shape,X_test.shape)

(506, 14) (354, 14) (152, 14)


In [18]:
print(X_test.head(),X_train.head())

         CRIM    ZN  INDUS  CHAS    NOX     RM   AGE     DIS   RAD    TAX  \
463   5.82115   0.0  18.10   0.0  0.713  6.513  89.9  2.8016  24.0  666.0   
152   1.12658   0.0  19.58   1.0  0.871  5.012  88.0  1.6102   5.0  403.0   
291   0.07886  80.0   4.95   0.0  0.411  7.148  27.7  5.1167   4.0  245.0   
183   0.10008   0.0   2.46   0.0  0.488  6.563  95.6  2.8470   3.0  193.0   
384  20.08490   0.0  18.10   0.0  0.700  4.368  91.2  1.4395  24.0  666.0   

     PTRATIO       B  LSTAT  MEDV  
463     20.2  393.82  10.29  20.2  
152     14.7  343.28  12.12  15.3  
291     19.2  396.90   3.56  37.3  
183     17.8  396.90   5.68  32.5  
384     20.2  285.83  30.63   8.8           CRIM    ZN  INDUS  CHAS    NOX     RM   AGE     DIS   RAD    TAX  \
485  3.67367   0.0  18.10   0.0  0.583  6.312  51.9  3.9917  24.0  666.0   
275  0.09604  40.0   6.41   0.0  0.447  6.854  42.8  4.2673   4.0  254.0   
155  3.53501   0.0  19.58   1.0  0.871  6.152  82.6  1.7455   5.0  403.0   
350  0.06211  40.

Linear Regression


In [19]:
lin_reg=LinearRegression()
lin_reg.fit(X_train.astype('float'),y_train.astype('float'))
y_predict=lin_reg.predict(X_test)
score1=metrics.r2_score(y_test,y_predict)
print(score1)
score2=metrics.mean_squared_error(y_test,y_predict)
print(score2)

1.0
1.0268685441251935e-28


Linear,ridge and lossy regression

In [20]:
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.metrics import mean_squared_error,r2_score

In [22]:
df.head()


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV,price
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2,36.2


In [24]:
X=df.drop('MEDV',axis=1)
# MEDV-> Median value of owner occupied homes
y=df['MEDV']

In [37]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
lr=LinearRegression()
lr.fit(X_train,y_train)
y_pred_lr=lr.predict(X_test)
print("linear reg")
print("MSE:",mean_squared_error(y_test,y_pred_lr))
print("R2 score:",r2_score(y_test,y_pred_lr))

linear reg
MSE: 1.5043500123260262e-27
R2 score: 1.0


In [32]:
print(X_test.dtypes)

CRIM        float64
ZN          float64
INDUS       float64
CHAS       category
NOX         float64
RM          float64
AGE         float64
DIS         float64
RAD        category
TAX         float64
PTRATIO     float64
B           float64
LSTAT       float64
price       float64
dtype: object


In [33]:
X_train=X_train.astype('float')
X_test=X_test.astype('float')

In [38]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
lr=LinearRegression()
lr.fit(X_train,y_train)
y_pred_lr=lr.predict(X_test)
print("linear reg")
print("MSE:",mean_squared_error(y_test,y_pred_lr))
print("R2 score:",r2_score(y_test,y_pred_lr))

linear reg
MSE: 1.5043500123260262e-27
R2 score: 1.0


Linear Regression coeeficients

In [40]:
lr_coeff=pd.Series(lr.coef_,index=X.columns)
print(lr_coeff)
# Note:linear regression is sensitive to multicollinearity and over fitting

CRIM      -9.092397e-17
ZN        -5.620504e-16
INDUS     -1.056555e-15
CHAS       2.004814e-15
NOX       -6.208216e-15
RM        -1.334260e-15
AGE        1.322727e-16
DIS       -6.528523e-16
RAD       -3.981190e-16
TAX       -3.469447e-17
PTRATIO    1.520594e-16
B         -4.753142e-16
LSTAT     -7.853961e-16
MEDV       1.000000e+00
dtype: float64


Ridge regression

In [41]:
ridge=Ridge(alpha=1.0)
ridge.fit(X_train,y_train)
y_pred_ridge=ridge.predict(X_test)
print("ridge reg")
print("MSE:",mean_squared_error(y_test,y_pred_ridge))
print("r2 score:",r2_score(y_test,y_pred_ridge))

ridge reg
MSE: 3.06383741346245e-07
r2 score: 0.9999999958220656


Ridge regression coefficients

In [42]:
ridge_coeff=pd.Series(ridge.coef_,index=X.columns)
print(ridge_coeff)

CRIM      -1.222110e-05
ZN         3.610441e-06
INDUS      8.388821e-07
CHAS       2.847869e-04
NOX       -1.067105e-03
RM         4.994884e-04
AGE       -1.363930e-06
DIS       -1.497737e-04
RAD        2.784487e-05
TAX       -1.283779e-06
PTRATIO   -9.270416e-05
B          1.414398e-06
LSTAT     -5.860639e-05
MEDV       9.998881e-01
dtype: float64


Lasso regression

In [43]:
lasso=Lasso(alpha=0.1)
lasso.fit(X_train,y_train)
y_pred_lasso=lasso.predict(X_test)
print("lasso reg")
print("MSE:",mean_squared_error(y_test,y_pred_lasso))
print("r2 score:",r2_score(y_test,y_pred_lasso))

lasso reg
MSE: 0.00011484672220661833
r2 score: 0.9999984339179399
